# qcap demo — seal, verify, tamper

Run this notebook in Colab with zero local install.

In [ ]:
import os
import shutil

# liboqs-python builds liboqs on first import (~2-5 min in Colab). Pin a known-good release.
os.environ["PYOQS_VERSION"] = "0.15.0"

# Build deps for liboqs (NOT the obsolete oqs-python PyPI package)
!apt-get -qq update >/dev/null
!apt-get -qq install -y cmake ninja-build build-essential libssl-dev git >/dev/null

if os.path.exists("/content/qcap"):
    shutil.rmtree("/content/qcap")
!git clone -q https://github.com/Nuqasm/Qcap.git /content/qcap
%cd /content/qcap

# Official wrapper: https://pypi.org/project/liboqs-python/
!pip install -q "liboqs-python>=0.14.1"
!pip install -q -e .

print("Loading liboqs (first import may take several minutes — please wait)...")
import oqs

mechanisms = oqs.get_enabled_sig_mechanisms()
pqc_ok = "ML-DSA-87" in mechanisms
print("liboqs loaded — ML-DSA-87 available:", pqc_ok)
if not pqc_ok:
    mldsa = [m for m in mechanisms if "ML-DSA" in m]
    print("ML-DSA mechanisms:", mldsa or "(none — will use Ed25519 fallback)")

In [ ]:
!qcap seal examples/model_custody/modelcard.json --out /tmp/llama-ft.qcap --signer alice
!qcap verify /tmp/llama-ft.qcap

In [ ]:
!printf 'x' >> /tmp/llama-ft.qcap
!qcap verify /tmp/llama-ft.qcap || echo 'tampered capsule correctly rejected'